# colab_05_eval — ONE-SHOT holdout eval on TEST (FREE, CPU)

**Anti-spin: TEST touched exactly once.** After this, no retraining. New ideas -> Future Work.

Outputs: `eval_report.json` + `fig05_pareto.png` + `DECISION_05.json`. Copy 3 files back to repo.

In [ ]:
# 0. Setup — self-recovering: finds repo, unzips upload, or clones (public or private via token)
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE = Path('/content/drive/MyDrive/Research/colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)
    print('Drive:', DRIVE)
except Exception as e:
    print('No Colab Drive (local run?):', e)
    DRIVE = Path('./colab_ckpt')
    DRIVE.mkdir(parents=True, exist_ok=True)

def _find_backend():
    """Return BACKEND dir (containing src/gateway/service.py) or None."""
    cands = [Path('/content/code-mixed-gateway/backend'), Path('/content/Research/backend'),
             Path('/content/backend'), Path.cwd()/'backend', Path.cwd()/'code-mixed-gateway'/'backend']
    try:
        for p in Path('/content').iterdir():
            if p.is_dir():
                cands += [p, p/'backend']
    except Exception: pass
    for c in cands:
        if (c/'src'/'gateway'/'service.py').exists(): return c
    # deep search one level for upload_tmp style dirs
    try:
        for p in Path('/content').glob('*/**/service.py'):
            if p.parent.parent.name == 'gateway' and p.parent.name == 'gateway':
                be = p.parent.parent.parent  # .../src/gateway/service.py -> backend?
                # walk up to dir containing 'src'
                q = p.parent
                while q != Path('/content') and q.name != 'src': q = q.parent
                be = q.parent
                if (be/'src'/'gateway'/'service.py').exists(): return be
    except Exception: pass
    return None

def _try_unzip_uploads():
    roots = [Path('/content')]
    try: roots.append(DRIVE)
    except Exception: pass
    for root in roots:
        try: zips = list(root.glob('backend*.zip'))
        except Exception: continue
        for z in zips:
            dest = Path('/content/repo_upload')
            if (dest/'backend'/'src'/'gateway'/'service.py').exists(): return True
            print(f'unzipping {z} -> {dest} ...')
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf: zf.extractall(dest)
            # handle zip containing backend/ at top or nested one level
            if not (dest/'backend'/'src'/'gateway'/'service.py').exists():
                for sub in dest.iterdir():
                    if (sub/'backend'/'src'/'gateway'/'service.py').exists():
                        import shutil
                        shutil.move(str(sub/'backend'), str(dest/'backend_tmp'))
                        shutil.move(str(dest/'backend_tmp'), str(dest/'backend'))
                        break
            print('unzip done')
            return True
    return False

def _try_clone():
    dest = Path('/content/code-mixed-gateway')
    if dest.exists(): return dest.exists()
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''
    except Exception: token = os.environ.get('GITHUB_TOKEN','') or ''
    url = ('https://'+token+'@github.com/tusharsaharan/code-mixed-gateway.git') if token else 'https://github.com/tusharsaharan/code-mixed-gateway.git'
    print('cloning (token=%s) ...' % ('yes' if token else 'no'))
    r = subprocess.run(['git','clone',url,str(dest)], capture_output=True, text=True)
    print('clone rc=', r.returncode)
    if r.returncode != 0: print((r.stderr or '')[-500:])
    return dest.exists()

BACKEND = _find_backend()
if BACKEND is None and _try_unzip_uploads(): BACKEND = _find_backend()
if BACKEND is None and _try_clone(): BACKEND = _find_backend()
print('BACKEND =', BACKEND)
assert BACKEND is not None, ('REPO NOT FOUND. Fastest fix (2 min): on your PC zip the backend folder '
    '(Research/backend -> backend.zip), drag backend.zip into Colab Files panel (folder icon, left), then '
    'Cell > Run this cell again. Alternative: make the GitHub repo public, or add a GITHUB_TOKEN secret.')
REPO = BACKEND.parent
DATA = BACKEND/'data'
print('REPO =', REPO, '| DATA exists:', DATA.exists())

%pip install -q pydantic pydantic-settings numpy httpx 2>&1 | tail -1
src = str(BACKEND/'src')
if src not in sys.path: sys.path.insert(0, src)
import gateway
print('import gateway OK from', gateway.__file__)


In [ ]:
# 1. Score TEST with heuristic + learned (rebuild features; no fitting on test)
import json, numpy as np, pickle
rows = [json.loads(l) for l in open(DRIVE/'calibration_real.jsonl', encoding='utf-8')]
by_id = {r['id']: r for r in rows}
te = splits['test']
h_te = np.array([by_id[i]['nonconformity'] for i in te])
y_te = np.array([0 if by_id[i]['cheap_success'] else 1 for i in te])
groups = [by_id[i]['group'] for i in te]
try:
    from sentence_transformers import SentenceTransformer
    from gateway.modules.m4_router.difficulty import DifficultyScorer
    lab = {json.loads(l)['id']: json.loads(l) for l in open(DRIVE/'groq_labels.jsonl', encoding='utf-8')}
    scorer = DifficultyScorer(); enc = SentenceTransformer('all-MiniLM-L6-v2')
    E = enc.encode([lab[i]['text'] for i in te], batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    F = []
    for k, rid in enumerate(te):
        f = scorer.features(lab[rid]['text'])
        F.append(np.concatenate([np.array([f.code_mix_ratio, f.entity_density, min(1,f.math_marker_count/3), min(1,f.char_count/400), len(lab[rid]['text'].split())/50], dtype=np.float32), E[k]]))
    F = np.stack(F)
    lr = pickle.load(open(DRIVE/'logreg.pkl','rb'))
    p_te = lr.predict_proba(F)[:,1] if F.shape[1] == lr.coef_.shape[1] else h_te
except Exception as e:
    print('fallback heuristic:', e); p_te = h_te
from sklearn.metrics import roc_auc_score
print('TEST AUROC heuristic/learned:', round(roc_auc_score(y_te, h_te),4), round(roc_auc_score(y_te, p_te),4))

In [ ]:
# 2. Apply FROZEN thresholds; bootstrap CIs; price of safety
import numpy as np, json
from gateway.pricing import CHEAP_PER_1K_USD, PREMIUM_PER_1K_USD
arms = th['arms']
taus = {'heuristic_safe': (h_te, arms['heuristic']['tau']), 'learned_safe': (p_te, arms['learned']['tau']), 'pure_rl': (p_te, arms['pure_rl']['tau'])}
# Mondrian: per-group learned taus
mt = arms['mondrian']['taus']
cheap_m = np.array([1 if p < mt.get(g, arms['learned']['tau']) else 0 for p, g in zip(p_te, groups)])
def metrics(cheap_mask):
    err = float(((1-cheap_mask)*0 + cheap_mask*y_te).sum()/len(y_te))  # unconditional cheap-fail rate
    share = float(cheap_mask.mean())
    # cost: cheap tokens at cheap rate, premium at premium rate (proxy rates, disclose)
    toks = 400  # avg prompt+completion; replaced by real TokenCounter in repo eval
    cost = share*toks*CHEAP_PER_1K_USD/1000 + (1-share)*toks*PREMIUM_PER_1K_USD/1000
    base = toks*PREMIUM_PER_1K_USD/1000
    return {'error': round(err,4), 'cheap_share': round(share,4), 'saving': round((base-cost)/base,4)}
rep = {}
for name, (sc, t) in taus.items():
    rep[name] = metrics((sc < t).astype(int)); print(name, 'tau', t, rep[name])
rep['learned_mondrian'] = metrics(cheap_m); print('learned_mondrian', rep['learned_mondrian'])
# bootstrap 1000x for CIs on error/saving
rng = np.random.default_rng(42); B = 1000
for name in list(rep.keys()):
    mask = {'heuristic_safe': (h_te<taus['heuristic_safe'][1]), 'learned_safe': (p_te<taus['learned_safe'][1]), 'pure_rl': (p_te<taus['pure_rl'][1]), 'learned_mondrian': cheap_m}[name]
    errs, savs = [], []
    for _ in range(B):
        idx = rng.integers(0, len(te), len(te))
        m = {'error': float((mask[idx]*y_te[idx]).sum()/len(idx)), 'saving': 0}
        errs.append(m['error'])
    rep[name]['error_ci95'] = [round(float(np.quantile(errs,0.025)),4), round(float(np.quantile(errs,0.975)),4)]
rep['price_of_safety'] = round(rep['pure_rl']['saving'] - rep['learned_safe']['saving'],4)
rep['proxy'] = True; rep['test_n'] = len(te)
(DRIVE/'eval_report.json').write_text(json.dumps(rep, indent=2)); print(rep)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.figure(); [plt.scatter(v['error'], v['saving'], label=k) for k,v in rep.items() if isinstance(v, dict) and 'error' in v]
plt.xlabel('realized error'); plt.ylabel('saving vs all-premium'); plt.legend(); plt.tight_layout()
plt.savefig(str(DRIVE/'fig05_pareto.png'), dpi=120); print('saved fig05')
(DRIVE/'DECISION_05.json').write_text(json.dumps({'stage':'05_eval_done','test_n':len(te),'frozen_report':'eval_report.json','rule':'DO NOT retrain after this'}, indent=2))